# 각 파일에서 6가지 점수 데이터 불러오기

In [11]:
# -*- coding: utf-8 -*-
"""
EWS 종합 위험 스코어용 변수 통합 스크립트 (수정본)
==========================================

[수정 내역]
 1. minmax_by_year 중복곱셈(*100*100 -> 0~10000 스케일) 버그 수정
    -> minmax() 내부에서 이미 *100을 하므로, minmax_by_year에서는 추가로 곱하지 않음
 2. 최종 출력 시 모든 점수 컬럼을 소수점 둘째자리까지 반올림
 3. (참고) 회사명 결측 / 기준연도-회계년도 불일치는 코드 버그가 아니라
    데이터셋 간 표본범위 차이 및 재무자료 입수 시차(reporting lag) 때문입니다.
    -> 자세한 설명은 채팅 답변 참고
"""

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------------
# 0. 설정
# ------------------------------------------------------------------

OUT_PATH = os.path.join("22번 스코어 산출\통합_스코어_데이터.csv")

# True로 바꾸면 기업단위 변수(기업베타/부실확률/diff)도
# '연도별 횡단면 정규화' 대신 '10개 연도 전체 통합 min-max'를 사용합니다.
GLOBAL_MINMAX = False


def read_csv_safe(filename, **kwargs):
    """사업자등록번호를 문자열(10자리 zfill)로 안전하게 읽는 CSV 로더"""
    path = os.path.join(filename)
    df = pd.read_csv(path, dtype={"사업자등록번호": str}, **kwargs)
    if "사업자등록번호" in df.columns:
        df["사업자등록번호"] = df["사업자등록번호"].str.zfill(10)
    return df


def minmax(s: pd.Series) -> pd.Series:
    """NaN을 무시하는 0~100 스케일 min-max. max==min이면 50으로 처리(분산 0)."""
    mn, mx = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.where(s.notna(), 50.0, np.nan), index=s.index)
    return (s - mn) / (mx - mn) * 100


def minmax_by_year(df, value_col, year_col="연도"):
    """연도별(횡단면) 0~100 스케일 min-max.
    주의: minmax() 내부에서 이미 *100을 하므로 여기서 추가로 곱하면 안 됨 (중복곱셈 버그 수정됨)
    """
    return df.groupby(year_col)[value_col].transform(minmax)


# ------------------------------------------------------------------
# 1. 베이스 테이블: lifecycle_scored_yearly_minmax.csv
#    (사업자등록번호, 기준연도) 단위가 고유(unique) -> '기준연도'를 '연도'로 사용
# ------------------------------------------------------------------
lifecycle = read_csv_safe("20번. 기업 생애주기\lifecycle_scored_yearly_minmax.csv")

base = lifecycle.rename(columns={"기준연도": "연도"})[
    ["사업자등록번호", "연도", "회계년도", "생애주기_최종", "생애주기_점수", "부실라벨_ICR3년"]
].copy()

# ------------------------------------------------------------------
# 2. 마이클 포터 5F (산업 단위, 도소매업) -> 연도 기준 broadcast
# ------------------------------------------------------------------
porter = read_csv_safe("19번 마이클 포터\마이클 포터 5F_도소매업.csv")
porter = porter[["연도", "최종점수"]].copy()
porter["porter_5F_minmax"] = minmax(porter["최종점수"])   # 10개 연도 시계열 min-max
porter = porter[["연도", "porter_5F_minmax"]]

# ------------------------------------------------------------------
# 3. 산업충격민감도_OLS (산업 단위, 도소매업) -> 연도 기준 broadcast
# ------------------------------------------------------------------
ind_ols = read_csv_safe("18번 산업별 충격민감도\산업충격민감도_OLS.csv")
ind_ols = ind_ols.rename(columns={"테스트_연도": "연도"})[["연도", "beta_i"]].copy()
ind_ols["산업베타_minmax"] = minmax(ind_ols["beta_i"])    # 10개 연도 시계열 min-max
ind_ols = ind_ols[["연도", "산업베타_minmax"]]

# ------------------------------------------------------------------
# 4. 충격민감도_OLS (기업 단위) -> (사업자등록번호, 연도) 기준
# ------------------------------------------------------------------
firm_ols = read_csv_safe("17번 기업별 충격민감도\충격민감도_OLS.csv")
firm_ols = firm_ols.rename(columns={"테스트_연도": "연도"})[
    ["사업자등록번호", "회사명", "연도", "beta_i"]
].copy()

if GLOBAL_MINMAX:
    firm_ols["기업베타_minmax"] = minmax(firm_ols["beta_i"])
else:
    firm_ols["기업베타_minmax"] = minmax_by_year(firm_ols, "beta_i")

firm_ols_for_merge = firm_ols[["사업자등록번호", "연도", "기업베타_minmax"]]
firm_name_map = firm_ols[["사업자등록번호", "회사명"]].drop_duplicates(subset=["사업자등록번호"])

# ------------------------------------------------------------------
# 5. 부실확률 차이값 (기업 단위, wide -> long 변환)
#    연도 범위: 2015~2024 (diff_2014_2015 ~ diff_2023_2024 이용)
# ------------------------------------------------------------------
prob = read_csv_safe(r"21번. 기업 PD 변화율\2015-2024_기업_부실확률_차이값.csv")

years = range(2015, 2025)
long_rows = []
for y in years:
    prob_col = f"prob_{y}"
    diff_col = f"diff_{y-1}_{y}"
    tmp = prob[["사업자등록번호", "회사명", prob_col, diff_col]].copy()
    tmp.columns = ["사업자등록번호", "회사명", "prob", "diff"]
    tmp["연도"] = y
    long_rows.append(tmp)

prob_long = pd.concat(long_rows, ignore_index=True)

if GLOBAL_MINMAX:
    prob_long["부실확률_minmax"] = minmax(prob_long["prob"])
    prob_long["부실확률변화_minmax"] = minmax(prob_long["diff"])
else:
    prob_long["부실확률_minmax"] = minmax_by_year(prob_long, "prob")
    prob_long["부실확률변화_minmax"] = minmax_by_year(prob_long, "diff")

prob_for_merge = prob_long[
    ["사업자등록번호", "회사명", "연도", "부실확률_minmax", "부실확률변화_minmax"]
]

# ------------------------------------------------------------------
# 6. 전체 병합
# ------------------------------------------------------------------
df = base.copy()

# 6-1. 산업 단위 변수 (연도 기준 broadcast)
df = df.merge(porter, on="연도", how="left")
df = df.merge(ind_ols, on="연도", how="left")

# 6-2. 기업 단위 변수
df = df.merge(firm_ols_for_merge, on=["사업자등록번호", "연도"], how="left")
df = df.merge(prob_for_merge, on=["사업자등록번호", "연도"], how="left", suffixes=("", "_prob"))

# 회사명 채우기: 부실확률 파일의 회사명을 우선 사용하고, 없는 경우만 충격민감도_OLS 매핑으로 보강
# (※ if/else로 분기하면, df에 '회사명'이 아직 없을 때 merge 결과 컬럼명이 '회사명_prob'가 아닌
#    '회사명'으로 들어가 else 분기가 타면서 기존 값을 덮어써버리는 버그가 있어 -> 항상 combine_first로 통일)
df["회사명"] = df["회사명"].combine_first(
    df["사업자등록번호"].map(firm_name_map.set_index("사업자등록번호")["회사명"])
)

# ------------------------------------------------------------------
# 6-3. 최종 출력용 컬럼명으로 변경
# ------------------------------------------------------------------
RENAME_MAP = {
    "porter_5F_minmax": "Porter5F",
    "생애주기_점수": "생애주기점수",
    "산업베타_minmax": "산업충격민감도",
    "기업베타_minmax": "기업충격민감도",
    "부실확률_minmax": "부실확률",
    "부실확률변화_minmax": "부실확률변화",
}
df = df.rename(columns=RENAME_MAP)

# ------------------------------------------------------------------
# 7. 최종 컬럼 정리, 소수점 둘째자리 반올림 및 저장
# ------------------------------------------------------------------
final_cols = [
    "사업자등록번호", "회사명", "연도", 
    "생애주기_최종", "부실라벨_ICR3년",
    "Porter5F",          
    "생애주기점수",        
    "산업충격민감도",      
    "기업충격민감도",     
    "부실확률",            
    "부실확률변화",       
]

df_final = df[final_cols].sort_values(["사업자등록번호", "연도"]).reset_index(drop=True)

# 점수 컬럼만 소수점 둘째자리로 반올림
score_cols = [
    "생애주기점수", "Porter5F", "산업충격민감도",
    "기업충격민감도", "부실확률", "부실확률변화",
]
df_final[score_cols] = df_final[score_cols].round(2)

# ------------------------------------------------------------------
# 7-1. '부실확률'이 없는 행 삭제
# ------------------------------------------------------------------
before_n = len(df_final)
df_final = df_final.dropna(subset=["부실확률"]).reset_index(drop=True)
after_n = len(df_final)
print(f"\n'부실확률' 결측 행 삭제: {before_n} -> {after_n} ({before_n - after_n}행 제거)")

df_final.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"완료: {OUT_PATH}")
print(f"행/열: {df_final.shape}")
print(df_final.head(10))
print("\n[결측치 비율]")
print(df_final.isna().mean().round(3))
print("\n[점수 컬럼 범위 확인 - 정상적으로 0~100 사이여야 함]")
print(df_final[score_cols].describe().loc[['min','max']])

<>:58: SyntaxWarning: invalid escape sequence '\l'
<>:58: SyntaxWarning: invalid escape sequence '\l'
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15516\2687088149.py:58: SyntaxWarning: invalid escape sequence '\l'
  lifecycle = read_csv_safe("20번. 기업 생애주기\lifecycle_scored_yearly_minmax.csv")



'부실확률' 결측 행 삭제: 48057 -> 32801 (15256행 제거)
완료: 22번 스코어 산출\통합_스코어_데이터.csv
행/열: (32801, 11)
      사업자등록번호          회사명    연도 생애주기_최종  부실라벨_ICR3년  Porter5F  생애주기점수  \
0  0008110041  엘브이엠씨홀딩스(주)  2015     도입기           0    100.00    75.7   
1  0008110041  엘브이엠씨홀딩스(주)  2016     도입기           0     56.26    80.6   
2  0008110041  엘브이엠씨홀딩스(주)  2017     성장기           0     34.84     1.1   
3  0008110041  엘브이엠씨홀딩스(주)  2018     성장기           0     25.54     4.0   
4  0008110041  엘브이엠씨홀딩스(주)  2019     성장기           0     56.69     0.2   
5  0008110041  엘브이엠씨홀딩스(주)  2020     성장기           1      0.00     0.0   
6  1018106586      서원물산(주)  2015     도입기           1    100.00    75.7   
7  1018116269   현대코퍼레이션(주)  2015     성숙기           0    100.00     3.2   
8  1018116269   현대코퍼레이션(주)  2016     성장기           0     56.26     0.3   
9  1018116269   현대코퍼레이션(주)  2017     조정기           0     34.84    21.8   

   산업충격민감도  기업충격민감도   부실확률  부실확률변화  
0     0.00    44.36   0.19   49.99  
1    42.55    48.52 

# 가중치 곱해 종합점수 산출

In [29]:
import pandas as pd

score_df = pd.read_csv("22번 스코어 산출/통합_스코어_데이터.csv")

other_cols = ["Porter5F", "생애주기점수", "산업충격민감도", "기업충격민감도", "부실확률변화"]

# 부실확률 가중치 후보: 0.4, 0.5, 0.6, 0.7, 0.8
prob_weights = [0.2, 0.3,0.4, 0.5, 0.6, 0.7, 0.8]

for w in prob_weights:
    other_w = (1 - w) / len(other_cols)  # 나머지 5개 변수에 동일 분배

    col_name = f"종합점수_{w}"
    score_df[col_name] = (
        score_df["부실확률"] * w
        + score_df[other_cols].sum(axis=1) * other_w
    ).round(2)

score_df.to_csv("22번 스코어 산출/통합_스코어_데이터.csv", index=False, encoding="utf-8-sig")

# 신용등급 존재하는 데이터 merge

In [30]:
import pandas as pd
import re

credit_df = pd.read_excel(r"..\데이터수집\신용등급\상장사 신용등급.xlsx")

# ============================================================
# 2. 신용등급 파일 전처리
# ============================================================
credit_df["사업자등록번호"] = (
    credit_df["사업자등록번호"]
    .astype(str).str.strip()
    .str.replace("-", "", regex=False)
    .astype("int64")
)

# 회계년도: "2015/12" -> 연도=2015, 월=12
credit_df["연도"] = credit_df["회계년도"].astype(str).str.split("/").str[0].astype(int)
credit_df["월"]   = credit_df["회계년도"].astype(str).str.split("/").str[1].astype(int)

# 평가사구분 10(NICE신용평가)만 사용
credit_filtered = credit_df[credit_df["평가사구분"].isin([10])].copy()




# ------------------------------------------------------------
# 2-2. 장기등급만 필터링 (단기CP등급 제외)
#   - 장기등급: AAA, AA+, AA, AA-, A+, A, A-, BBB+, ... (등급 뒤에 /STABLE 등 전망이 붙을 수 있음)
#   - 단기CP등급: A1, A2+, A2, A2-, A3+, A3, A3-, B+, B0, B- 등 -> 등급 표기에 숫자가 포함됨
# ------------------------------------------------------------
def is_long_term(rating):
    if pd.isna(rating):
        return False
    grade = str(rating).split("/")[0]  # "/STABLE" 같은 전망 부분 제거
    # 등급 표기에 숫자가 있으면 단기(CP)등급으로 판단
    return not re.search(r"\d", grade)

credit_filtered = credit_filtered[credit_filtered["신용등급"].apply(is_long_term)].copy()

print("필터링 후 신용등급 unique 값:")
print(sorted(credit_filtered["신용등급"].dropna().unique()))


# ------------------------------------------------------------
# 2-3. 연중 변경 시 -> 연말(가장 최근 월) 등급 채택
# ------------------------------------------------------------
# 동일 (사업자등록번호, 연도) 내에서 '월'이 가장 큰 행만 남김
idx_latest = (
    credit_filtered
    .groupby(["사업자등록번호", "연도"])["월"]
    .idxmax()
)
credit_filtered = credit_filtered.loc[idx_latest].copy()


# ------------------------------------------------------------
# 2-4. 동일 (사업자등록번호, 연도) 내에서, 같은 월에 등급이 여러개면
#      등급(전망 제외 base grade) 기준으로 동일 여부 판단
# ------------------------------------------------------------
credit_filtered["등급_base"] = credit_filtered["신용등급"].str.split("/").str[0]

credit_filtered = credit_filtered.drop_duplicates(
    subset=["사업자등록번호", "연도", "월", "평가사명 및 등급", "신용등급"]
)

def collapse_if_same(group):
    unique_base = group["등급_base"].dropna().unique()
    if len(unique_base) <= 1:
        return group.iloc[[0]]
    else:
        biz_no = group["사업자등록번호"].iloc[0]
        year = group["연도"].iloc[0]
        print(f"\n[신용등급 불일치] 사업자등록번호: {biz_no}, 연도: {year}")
        print(group[["월", "신용등급", "평가사명 및 등급"]].to_string(index=False))
        return group

credit_filtered = (
    credit_filtered
    .groupby(["사업자등록번호", "연도"], group_keys=False)
    .apply(collapse_if_same)
    .reset_index(drop=True)
)

credit_for_merge = credit_filtered[
    ["사업자등록번호", "연도", "평가사명 및 등급", "신용등급", "평가사구분"]
]


# ============================================================
# 3. 병합
# ============================================================
result_df = score_df.merge(credit_for_merge, on=["사업자등록번호", "연도"], how="inner")


# ============================================================
# 4. 최종 컬럼 정리
# ============================================================
final_cols = [
    "사업자등록번호", "회사명", "연도",
    "종합점수_0.2","종합점수_0.3","종합점수_0.4", "종합점수_0.5", "종합점수_0.6", "종합점수_0.7", "종합점수_0.8",
    "평가사명 및 등급", "신용등급", "평가사구분",
]
result_df = result_df[final_cols]

print(f"\n결과 shape: {result_df.shape}")
print(f"고유 (사업자등록번호, 연도) 수: {result_df[['사업자등록번호','연도']].drop_duplicates().shape[0]}")
print(result_df.head(10))

result_df.to_csv(r"22번 스코어 산출\통합_스코어_신용등급_병합.csv", index=False, encoding="utf-8-sig")

필터링 후 신용등급 unique 값:
['-', 'A', 'A ', 'A (긍정적)', 'A (부정적)', 'A (안정적)', 'A / STABLE', 'A(Positive)', 'A(STABLE)', 'A(긍정적)', 'A(부정적)', 'A(안정적)', 'A(하향검토)', 'A+', 'A+ (긍정적)', 'A+ (부정적)', 'A+ (안정적)', 'A+ (하향검토)', 'A+ Positive', 'A+ Stable', 'A+(POSITIVE)', 'A+(Positive)', 'A+(STABLE)', 'A+(부정적)', 'A+(안정적)', 'A+(하향검토)', 'A+/ 부정적', 'A+/ 안정적', 'A+/STABLE', 'A+/긍정적', 'A+/안정적', 'A+STABLE', 'A+↓', 'A-', 'A-  (긍정적)', 'A-  (안정적)', 'A- (긍정적)', 'A- (부정적)', 'A- (안정적)', 'A- / POSITIVE', 'A- / Positive', 'A- STABLE', 'A- sTABLE', 'A-(Positive)', 'A-(STABLE)', 'A-(Stable)', 'A-(긍정적)', 'A-(부정적)', 'A-(안정적)', 'A-(하향검토)', 'A-/Positive', 'A-/STABLE', 'A-/부정적', 'A-/안정적', 'A-↓', 'A/ 안정적', 'A/Positive', 'A/Postive', 'A/STABLE', 'A/부정적', 'A/안정적', 'AA', 'AA (긍정적)', 'AA (부정적)', 'AA (안정적)', 'AA 안정적', 'AA(STABLE)', 'AA(긍정적)', 'AA(부정적)', 'AA(안정적)', 'AA+', 'AA+ (긍정적)', 'AA+ (부정적)', 'AA+ (안정적)', 'AA+(Negative)', 'AA+(Positive)', 'AA+(STABLE)', 'AA+(긍정적)', 'AA+(부정적)', 'AA+(안정적)', 'AA+/N', 'AA+/안정적', 'AA+sTABLE', 'AA+sta

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15516\347546654.py:80: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(collapse_if_same)


In [31]:
import pandas as pd
import re
from scipy.stats import spearmanr, kendalltau

df = pd.read_csv("22번 스코어 산출\통합_스코어_신용등급_병합.csv")

# ============================================================
# 1. 신용등급 표기 정리
#    예: "AA-/STABLE", "AA-(STABLE)", "AA-(안정적)", "AA+sTABLE", "B(싱글비)" -> "AA-", "AA-", "AA-", "AA+", "B"
# ============================================================
def clean_grade(x):
    x = str(x)
    x = x.split("/")[0]                 # "AA-/STABLE" -> "AA-"
    x = re.sub(r"\(.*?\)", "", x)       # "AA-(안정적)" -> "AA-"
    x = re.sub(r"[A-Za-z]*[Ss][Tt][Aa][Bb][Ll][Ee]", "", x)  # "AA+sTABLE" -> "AA+"
    return x.strip()

df["등급_clean"] = df["신용등급"].apply(clean_grade)

print("정리된 등급 종류:")
print(sorted(df["등급_clean"].unique()))
print(df["등급_clean"].value_counts())


# ============================================================
# 2. 신용등급 -> 순위(ordinal) 변환
#    숫자가 작을수록 우량 (AAA=1, AA+=2, ... BB-=13 ...)
# ============================================================
order = ["AAA","AA+","AA","AA-","A+","A","A-","BBB+","BBB","BBB-",
         "BB+","BB","BB-","B+","B","B-","CCC+","CCC","CCC-","CC","C","D"]
grade_map = {g: i + 1 for i, g in enumerate(order)}

df["등급_순위"] = df["등급_clean"].map(grade_map)


# ============================================================
# 3. 종합점수(0.4~0.8) vs 신용등급_순위 : 상관관계 계산
# ============================================================
score_cols = ["종합점수_0.2","종합점수_0.3","종합점수_0.4", "종합점수_0.5", "종합점수_0.6", "종합점수_0.7", "종합점수_0.8"]

print("\n컬럼명 / Spearman / p-value / Kendall")
for col in score_cols:
    rho, p = spearmanr(df[col], df["등급_순위"])
    tau, p2 = kendalltau(df[col], df["등급_순위"])
    print(f"{col}: spearman={rho:.4f} (p={p:.4f}), kendall={tau:.4f} (p={p2:.4f})")

정리된 등급 종류:
['A', 'A+', 'A-', 'AA', 'AA+', 'AA-', 'B', 'BB-', 'BBB', 'BBB+', 'BBB-']
등급_clean
AA-     59
AA+     31
A-      24
AA      24
A       19
A+      15
BB-      4
B        3
BBB      3
BBB-     2
BBB+     1
Name: count, dtype: int64

컬럼명 / Spearman / p-value / Kendall
종합점수_0.2: spearman=0.1630 (p=0.0266), kendall=0.1178 (p=0.0272)
종합점수_0.3: spearman=0.1371 (p=0.0627), kendall=0.0972 (p=0.0685)
종합점수_0.4: spearman=0.1220 (p=0.0980), kendall=0.0879 (p=0.0993)
종합점수_0.5: spearman=0.0898 (p=0.2243), kendall=0.0655 (p=0.2194)
종합점수_0.6: spearman=0.0635 (p=0.3904), kendall=0.0465 (p=0.3832)
종합점수_0.7: spearman=0.0424 (p=0.5663), kendall=0.0345 (p=0.5182)
종합점수_0.8: spearman=0.0350 (p=0.6359), kendall=0.0309 (p=0.5623)


# 그리드 서치했을때의 최적 가중치

In [33]:
# ============================================================
# 4. 가중치 조합 생성 (0.05 단위, 모든 변수 최소 5%)
#    - 부실확률 가중치 w_pd: 0.20 ~ 0.75 (0.05 단위)
#      -> 나머지 5개 변수가 각각 최소 0.05를 가지려면
#         (1 - w_pd) >= 0.25 이어야 하므로 w_pd <= 0.75
#    - 나머지 5개 변수: 각자 최소 1단위(0.05)를 먼저 배정한 뒤,
#      남는 단위를 0.05 단위로 추가 분배 (합 = 1 - w_pd 보장)
# ============================================================
def gen_combos(total_units, n_vars):
    """0.05 단위(=total_units개)를 n_vars개 변수에 비음수로 분배하는 모든 조합 생성"""
    if n_vars == 1:
        yield (total_units,)
        return
    for i in range(total_units + 1):
        for rest in gen_combos(total_units - i, n_vars - 1):
            yield (i,) + rest


# ============================================================
# 5. 그리드서치: 모든 가중치 조합에 대해 Spearman 계산
# ============================================================
y = merged["등급_순위"].values
other_cols = ["Porter5F", "생애주기점수", "산업충격민감도", "기업충격민감도", "부실확률변화"]
n_other = len(other_cols)

results = []

# w_pd: 0.20(=4*0.05) ~ 0.75(=15*0.05)
# (나머지 5개 변수가 각자 최소 1단위(0.05)를 갖기 위해 최대 0.75까지만 허용)
for pd_units in range(4, 16):
    w_pd = round(pd_units * 0.05, 2)

    # 부실확률을 뺀 나머지 단위 합
    remaining_total = 20 - pd_units

    # 5개 변수에 각자 최소 1단위를 먼저 배정 -> 추가로 분배할 단위 수
    extra_units = remaining_total - n_other  # >= 0 (w_pd<=0.75에서 항상 성립)

    for extra_combo in gen_combos(extra_units, n_other):
        # 각 변수 = (기본 1단위 + 추가단위) * 0.05
        weights_other = [round((1 + e) * 0.05, 2) for e in extra_combo]

        score = merged[pd_col].values * w_pd
        for col, w in zip(other_cols, weights_other):
            score = score + merged[col].values * w

        rho, p = spearmanr(score, y)

        results.append({
            "부실확률_w": w_pd,
            **{f"{c}_w": w for c, w in zip(other_cols, weights_other)},
            "spearman_rho": rho,
            "p_value": p,
        })

res_df = pd.DataFrame(results)


# ============================================================
# 6. 결과 정렬 및 출력
# ============================================================
res_df_sorted = res_df.sort_values("p_value")

print(f"\n총 탐색 조합 수: {len(res_df)}")
print("\n[p-value 기준 상위 10개 조합]")
print(res_df_sorted.head(10).to_string(index=False))

print(f"\n5% 기준 우연히 p<0.05가 나올 것으로 예상되는 조합 수: 약 {int(len(res_df)*0.05)}개")
print(f"실제 p<0.05인 조합 수: {(res_df['p_value'] < 0.05).sum()}개")

res_df_sorted.to_csv("22번 스코어 산출/가중치_그리드서치_결과.csv", index=False, encoding="utf-8-sig")


총 탐색 조합 수: 4368

[p-value 기준 상위 10개 조합]
 부실확률_w  Porter5F_w  생애주기점수_w  산업충격민감도_w  기업충격민감도_w  부실확률변화_w  spearman_rho  p_value
   0.20        0.05      0.45       0.15       0.05      0.10      0.244739 0.000787
   0.20        0.05      0.50       0.15       0.05      0.05      0.242893 0.000864
   0.20        0.05      0.55       0.10       0.05      0.05      0.241782 0.000914
   0.20        0.10      0.40       0.20       0.05      0.05      0.241598 0.000923
   0.20        0.05      0.40       0.15       0.05      0.15      0.241216 0.000941
   0.25        0.05      0.45       0.15       0.05      0.05      0.240653 0.000968
   0.20        0.05      0.50       0.10       0.05      0.10      0.240335 0.000984
   0.20        0.10      0.40       0.15       0.05      0.10      0.240269 0.000987
   0.20        0.05      0.30       0.05       0.35      0.05      0.240195 0.000990
   0.20        0.05      0.50       0.10       0.10      0.05      0.240104 0.000995

5% 기준 우연히 p<0.05가 나올 것으